- []TF-IDF = Mencari sebuah kata yang sering muncul di dokument tertentu, namun tidak terlalu sering muncul di dokumen tertentu


TF -> Term Frequency -> Seberapa sering sebuah kata muncul di dokumen


IDF -> Inverse Document Frequency -> menunjukkan seberapa langka sbuah kata di seluruh kumpulan dokumen
- []Cosine Similarity


# Manual

In [2]:
import pandas as pd
import math
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import numpy as np
from numpy.linalg import norm

In [3]:
document_1 = 'Sally sells shiny silver seashells by the calm seashore evey sunny Saturday'
document_2 = 'Sally stacks slippery silver seashells by the windy seashore each cloudy sunday'

document_1 = word_tokenize(document_1.lower())
document_2 = word_tokenize(document_2.lower())

eng_stopwords = stopwords.words('english')

removed_stopwords_1 = [token for token in document_1 if token not in eng_stopwords]
removed_stopwords_2 = [token for token in document_2 if token not in eng_stopwords]

merge = set(removed_stopwords_1).union(set(removed_stopwords_2))

print(merge)


{'sally', 'seashells', 'evey', 'silver', 'cloudy', 'calm', 'saturday', 'sunny', 'windy', 'shiny', 'seashore', 'slippery', 'sunday', 'stacks', 'sells'}


In [4]:
# Membuat dictionary frequency {kata, freq}

word_dict_1 = dict.fromkeys(merge, 0)
word_dict_2 = dict.fromkeys(merge, 0)

for word in removed_stopwords_1:
    word_dict_1[word] += 1

for word in removed_stopwords_2:
    word_dict_2[word] += 1

pd.DataFrame([word_dict_1,word_dict_2])

,sally,seashells,evey,silver,cloudy,calm,saturday,sunny,windy,shiny,seashore,slippery,sunday,stacks,sells
0,1,1,1,1,0,1,1,1,0,1,1,0,0,0,1
1,1,1,0,1,1,0,0,0,1,0,1,1,1,1,0


In [5]:
# Menghitung TF -> term frequency
# Jumlah kata yg muncul di dokumen itu / jumlah kata di dokumen itu

def compute_TF(word_dict, doc):
    tf_dict = {}
    corpus_count = len(doc)
    for word, count in word_dict.items():
        tf_dict[word] = count / corpus_count

    return tf_dict

tf_1 = compute_TF(word_dict_1, removed_stopwords_1)
tf_2 = compute_TF(word_dict_2, removed_stopwords_2)


tf = pd.DataFrame([tf_1,tf_2])
tf

,sally,seashells,evey,silver,cloudy,calm,saturday,sunny,windy,shiny,seashore,slippery,sunday,stacks,sells
0,0.100000,0.100000,0.1,0.100000,0.000000,0.1,0.1,0.1,0.000000,0.1,0.100000,0.000000,0.000000,0.000000,0.1
1,0.111111,0.111111,0.0,0.111111,0.111111,0.0,0.0,0.0,0.111111,0.0,0.111111,0.111111,0.111111,0.111111,0.0


In [6]:
def compute_IDF(doclist):
    idf_dict = {}
    N = len(doclist)

    for doc in doclist:
        for word, count in doc.items():
            if count > 0:
                idf_dict[word] = idf_dict.get(word, 0) + 1

    for word, df in idf_dict.items():
        idf_dict[word] = math.log((N+1)/(df+1)) + 1

    idf_df = pd.DataFrame(list(idf_dict.items()), columns=['Word', 'IDF'])

    return idf_df

idfs = compute_IDF([word_dict_1, word_dict_2])
idfs

,Word,IDF
0,sally,1.000000
1,seashells,1.000000
2,evey,1.405465
3,silver,1.000000
4,calm,1.405465
5,saturday,1.405465
6,sunny,1.405465
7,shiny,1.405465
8,seashore,1.000000
9,sells,1.405465


In [8]:
def compute_TFIDF(tfs, idfs):
    tfidf = {}

    for word, val in tfs.items():
        tfidf[word] = val * idfs.loc[idfs['Word'] == word, 'IDF'].values[0]

    return tfidf

tfidf_1 = compute_TFIDF(tf_1, idfs)
tfidf_2 = compute_TFIDF(tf_2, idfs)


tfidf = pd.DataFrame([tfidf_1, tfidf_2])


tfidf

,sally,seashells,evey,silver,cloudy,calm,saturday,sunny,windy,shiny,seashore,slippery,sunday,stacks,sells
0,0.100000,0.100000,0.140547,0.100000,0.000000,0.140547,0.140547,0.140547,0.000000,0.140547,0.100000,0.000000,0.000000,0.000000,0.140547
1,0.111111,0.111111,0.000000,0.111111,0.156163,0.000000,0.000000,0.000000,0.156163,0.000000,0.111111,0.156163,0.156163,0.156163,0.000000


In [9]:
def cosine_similarity_manual(tfidf_1, tfidf_2):
    words = list(tfidf_1.keys())

    # mengambil tf idf utk setiap kata dan kita ubah ke dalam bentuk array
    vec1 = [tfidf_1[word] for word in words]
    vec2 = [tfidf_2[word] for word in words]

    # menghitung cosine similarity
    return np.dot(vec1, vec2) / (norm(vec1) * norm(vec2))

cosine = cosine_similarity_manual(tfidf_1, tfidf_2)
print(cosine)

0.2696966599826406


# Library

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
document_1 = 'Sally sells shiny silver seashells by the calm seashore evey sunny Saturday'
document_2 = 'Sally stacks slippery silver seashells by the windy seashore each cloudy sunday'

docs = [document_1, document_2]
print(docs)

['Sally sells shiny silver seashells by the calm seashore evey sunny Saturday', 'Sally stacks slippery silver seashells by the windy seashore each cloudy sunday']


In [12]:
vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(docs)

# mengubah yang awalnya matrix menjadi array
X1 = X.toarray()[0]
X2 = X.toarray()[1]

print('\nX1: ', X1)
print('\nX2: ', X2)


X1:  [0.35300279 0.         0.35300279 0.25116439 0.35300279 0.25116439
 0.25116439 0.35300279 0.35300279 0.25116439 0.         0.
 0.         0.35300279 0.        ]

X2:  [0.         0.37729199 0.         0.26844636 0.         0.26844636
 0.26844636 0.         0.         0.26844636 0.37729199 0.37729199
 0.37729199 0.         0.37729199]


In [13]:
cosine = np.dot(X1, X2) / (norm(X1) * norm(X2))
print(cosine)

0.26969665998264064


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(X)

sklearn_cosine = sim_matrix[0][1]

print(sklearn_cosine)

0.2696966599826407
